# Klint-32M v2: Institutional Financial Fine-Tuning Engine
### Upgrading Causal Autoregressive Foundation Models with Multi-Asset Data & PnL-Weighted Financial Loss
**Author**: Akhilesh Varma ([@akhverm](https://huggingface.co/akhverm)) | **Repository**: [ak495867/Klint-32M](https://github.com/ak495867/Klint-32M)

---

### Abstract & Financial Motivation
Standard foundation time-series models optimize symmetrical **Cross-Entropy** loss:
$$\mathcal{L}_{\text{CE}} = -\sum_{i} y_i \log \hat{y}_i$$

In financial markets, this induces severe trading pathology:
1. **Symmetric Loss Blindness**: A 2-pip error during a quiet sideways market is penalized the same as a 200-pip error during a flash crash.
2. **Directional Indifference**: Predicting $+0.05\%$ when the market moves $-0.05\%$ yields identical cross-entropy to predicting $+0.15\%$ when the market moves $+0.05\%$, yet one bankrupts a momentum portfolio while the other captures positive alpha.
3. **Single-Asset Overfitting**: Pre-training exclusively on one asset (e.g. SOL 1.59M bars) leads to market regime blindness when exposed to cross-market shifts in equities, commodities, or fixed income.

This notebook fine-tunes `klint_32m_release.pt` into **Klint-32M v2** using:
* **Multi-Asset Cross-Market Ingestion**: Equities (`SPY`, `QQQ`, `AAPL`, `NVDA`, `MSFT`, `TSLA`), Crypto (`BTC-USD`, `ETH-USD`, `SOL-USD`), Commodities (`GLD`, `USO`), and Rates (`TLT`).
* **PnL-Weighted Loss**: Scales loss dynamically by realized volatility magnitude:
  $$w_t = 1.0 + \lambda_{\text{pnl}} \cdot |r_t^{\text{body}}| \cdot 100$$
* **Asymmetric Directional Hinge Penalty**: Directly penalizes wrong-sign directional forecasts:
  $$\mathcal{L}_{\text{dir}} = \operatorname{ReLU}(-\hat{r}_{\text{pred}} \cdot r_{\text{realized}}) \cdot 100$$
* **Zero Google Drive Dependency**: Operates entirely in `./checkpoints/` with automated Hugging Face acquisition.


In [1]:
# Stage 1: Environment Setup & Hardware Verification
import os
import sys
import torch

print(f"PyTorch Version : {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Available  : {gpu_available}")

if gpu_available:
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Hardware    : {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    device = "cpu"
    print("Warning: Running on CPU. For fast training, enable a GPU runtime (Runtime > Change runtime type > T4 GPU).")

# Install dependencies if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("\n[Colab Detected] Installing required dependencies & cloning repo...")
    !pip install -q yfinance huggingface_hub matplotlib seaborn
    if not os.path.exists("src/klint"):
        !git clone -q https://github.com/ak495867/Klint-32M.git
        %cd Klint-32M
    sys.path.insert(0, os.path.abspath("src"))
    !pip install -q -e .
except ImportError:
    IN_COLAB = False
    sys.path.insert(0, os.path.abspath("src"))
    print("\n[Local / Dedicated Server] Using local environment.")

print("Environment setup complete.")


PyTorch Version : 2.11.0+cu128
CUDA Available  : True
GPU Hardware    : Tesla T4 (14.56 GB VRAM)

[Colab Detected] Installing required dependencies & cloning repo...
/content/Klint-32M
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for klint (pyproject.toml) ... done
Environment setup complete.


---
## Stage 2: Foundation Weights Acquisition (Warm-Start)
We load the pre-trained **Klint-32M** foundation weights ($28,642,560$ parameters).
* If `checkpoints/klint_32m_release.pt` is not present locally, it is automatically fetched directly from Hugging Face (`akhverm/Klint-32M`).
* Standalone model checkpoints (`checkpoints/klint_32m_best.pt` and `checkpoints/tokenizer_best.pt`) are automatically bundled if available.


In [2]:
# Stage 2: Load or Download Foundation Checkpoints
from klint.eval.bundle_loader import get_or_create_release_bundle
from klint.models.klint_32m import Klint32M, KlintConfig
from klint.tokenizer.factor_tokenizer import FactorTokenizer
from klint.tokenizer.geometric_decoder import GeometricDecoder

os.makedirs("checkpoints", exist_ok=True)
bundle_path = "checkpoints/klint_32m_release.pt"

print("Loading Klint-32M Foundation Model & RVQ Factor Tokenizer...")
model, tokenizer, decoder, cfg, meta = get_or_create_release_bundle(
    bundle_path=bundle_path,
    model_checkpoint="checkpoints/klint_32m_best.pt",
    tokenizer_checkpoint="checkpoints/tokenizer_best.pt",
    hf_repo="akhverm/Klint-32M",
    device=device,
)

param_count = model.count_parameters()
print(f"\nModel Architecture Loaded Successfully:")
print(f"  * Total Trainable Parameters : {param_count:,}")
print(f"  * Hidden Dimension (d_model) : {cfg.d_model}")
print(f"  * Transformer Layers         : {cfg.n_layers}")
print(f"  * Attention Heads            : {cfg.n_heads}")
print(f"  * Price Vocab Size           : {cfg.price_vocab_size}")
print(f"  * Source Training Meta       : {meta}")


Loading Klint-32M Foundation Model & RVQ Factor Tokenizer...
Local checkpoints not found. Downloading release bundle from Hugging Face (akhverm/Klint-32M)...


klint_32m_release.pt: reconstructing file:   0%|          |  0.00B /  115MB            

klint_32m_release.pt: downloading bytes:           |  0.00B            

Loading all-in-one release bundle from: /root/.cache/huggingface/hub/models--akhverm--Klint-32M/snapshots/cf1c8e39ca96a3115fda8262a823ff32b521c582/klint_32m_release.pt

Model Architecture Loaded Successfully:
  * Total Trainable Parameters : 28,642,560
  * Hidden Dimension (d_model) : 480
  * Transformer Layers         : 10
  * Attention Heads            : 10
  * Price Vocab Size           : 512
  * Source Training Meta       : {}


---
## Stage 3: Multi-Asset Cross-Market Ingestion Engine
To prevent single-asset overfitting and instill cross-market regime awareness, we train Klint across 12 diverse institutional assets spanning four major asset classes:
1. **US Equities**: `SPY` (S&P 500), `QQQ` (Nasdaq 100), `AAPL`, `NVDA`, `MSFT`, `TSLA`
2. **Crypto Macro**: `BTC-USD`, `ETH-USD`, `SOL-USD`
3. **Commodities**: `GLD` (Gold), `USO` (Crude Oil)
4. **Rates / Fixed Income**: `TLT` (20+ Year Treasury Bonds)

Each time series is sanitized to strictly preserve financial invariants:
$$High \ge \max(Open, Close) \quad \text{and} \quad Low \le \min(Open, Close)$$
Then decomposed into causal factors ($r_{\text{gap}}, r_{\text{body}}, \log\text{range}, \text{wick ratios}, \text{rel vol}$) and quantized into discrete RVQ factor tokens.


In [3]:
# Stage 3: Ingest Multi-Asset Market Data & Build Dataloaders
from klint.finetune.multi_asset_dataset import (
    MultiAssetFineTuneDataset,
    build_finetune_dataloaders,
    DEFAULT_FINETUNE_TICKERS,
)

print("Target Multi-Asset Universe:")
for i, t in enumerate(DEFAULT_FINETUNE_TICKERS, 1):
    print(f"  {i:2d}. {t}")

# Look for any existing local datasets
local_files = [f for f in ["data/SOL.npy"] if os.path.exists(f)]
if local_files:
    print(f"Found local dataset: {local_files}")

# Build training and validation dataloaders
# Context window: 256 bars = 768 factor tokens (Price, Range, Activity)
print("\nFetching historical OHLCV data & quantizing into RVQ factor sequences...")
train_loader, val_loader = build_finetune_dataloaders(
    tickers=DEFAULT_FINETUNE_TICKERS,
    local_files=local_files,
    tokenizer=tokenizer,
    batch_size=16 if device == 'cuda' else 4,
    context_bars=256,
    stride_bars=64,
    period="2y",
    interval="1h",
    val_ratio=0.15,
)

print(f"\nDataLoader Summary:")
print(f"  * Training Batches   : {len(train_loader)}")
print(f"  * Validation Batches : {len(val_loader)}")


Target Multi-Asset Universe:
   1. AAPL
   2. MSFT
   3. NVDA
   4. GOOGL
   5. AMZN
   6. META
   7. TSLA
   8. AVGO
   9. AMD
  10. QCOM
  11. INTC
  12. CRM
  13. ORCL
  14. ADBE
  15. PLTR
  16. CSCO
  17. JPM
  18. V
  19. MA
  20. BAC
  21. WFC
  22. MS
  23. GS
  24. BLK
  25. COIN
  26. LLY
  27. UNH
  28. JNJ
  29. ABBV
  30. MRK
  31. PFE
  32. HD
  33. COST
  34. WMT
  35. PG
  36. KO
  37. CAT
  38. GE
  39. XOM
  40. CVX
  41. SPY
  42. QQQ
  43. IWM
  44. DIA
  45. VOO
  46. VTI
  47. XLK
  48. XLF
  49. XLV
  50. XLE
  51. XLI
  52. XLY
  53. XLP
  54. XLU
  55. XLB
  56. SMH
  57. SOXX
  58. ARKK
  59. EEM
  60. INDA
  61. BTC-USD
  62. ETH-USD
  63. SOL-USD
  64. BNB-USD
  65. XRP-USD
  66. ADA-USD
  67. DOGE-USD
  68. AVAX-USD
  69. LINK-USD
  70. DOT-USD
  71. NEAR-USD
  72. ATOM-USD
  73. LTC-USD
  74. BCH-USD
  75. XLM-USD
  76. GLD
  77. SLV
  78. USO
  79. UNG
  80. DBC
  81. CPER
  82. PPLT
  83. WEAT
  84. CORN
  85. SOYB
  86. TLT
  87. IEF
  88. SHY
  89. BND

/content/Klint-32M/src/klint/benchmark/data_fetcher.py:49: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
/content/Klint-32M/src/klint/benchmark/data_fetcher.py:49: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df = pd.read_csv(cache_file, index_col=0, parse_dates=True)


  --> Progress: 25/100 processed (26 active datasets)
  --> Progress: 50/100 processed (51 active datasets)
  --> Progress: 75/100 processed (76 active datasets)
  --> Progress: 100/100 processed (101 active datasets)
MultiAssetFineTuneDataset (val): Created 4409 sequence windows across 101 assets.

DataLoader Summary:
  * Training Batches   : 1800
  * Validation Batches : 276


---
## Stage 4: Financial Utility & Directional Hinge Loss
We configure the custom financial loss function:
$$\mathcal{L}_{\text{total}} = \frac{\mathcal{L}_{\text{price}} + \mathcal{L}_{\text{range}} + \mathcal{L}_{\text{activity}}}{3}$$

### 1. PnL-Weighted Cross-Entropy
$$\mathcal{L}_{\text{weighted}} = \frac{1}{N}\sum_{t=1}^N \left(1.0 + \lambda_{\text{pnl}} \cdot \min(|r_t^{\text{body}}| \cdot 100, 10.0)\right) \cdot \ell_{\text{CE}}(t)$$

### 2. Asymmetric Directional Penalty (Hinge Loss)
$$\hat{r}_{\text{pred}} = \sum_{c=0}^{511} P(c \mid \mathbf{x}) \cdot r_{\text{body}}(c)$$
$$\mathcal{L}_{\text{dir}} = \frac{1}{N}\sum_{t=1}^N \operatorname{ReLU}\left(-\hat{r}_{\text{pred}}(t) \cdot r_{\text{realized}}(t)\right) \cdot 100$$
$$\mathcal{L}_{\text{price}} = \mathcal{L}_{\text{weighted}} + \gamma_{\text{dir}} \cdot \mathcal{L}_{\text{dir}}$$


In [4]:
# Stage 4: Initialize PnL-Weighted Loss Function
from klint.finetune.loss import PnLWeightedCrossEntropyLoss

# lambda_pnl=2.0 puts 3x weight on 2% moves compared to flat bars
# gamma_dir=1.0 heavily penalizes opposite-sign predictions
criterion = PnLWeightedCrossEntropyLoss(
    tokenizer=tokenizer,
    lambda_pnl=2.0,
    gamma_dir=1.0,
    max_weight_clip=10.0,
).to(device)

print(f"PnL Loss Engine Initialized:")
print(f"  * Price Codebook Tokens Cached : {len(criterion.code_returns)}")
print(f"  * PnL Lambda (Volatility Scale): {criterion.lambda_pnl}")
print(f"  * Directional Gamma (Sign Hinge): {criterion.gamma_dir}")
print(f"  * Sample Body Return Min/Max   : [{criterion.code_returns.min().item():.4f}, {criterion.code_returns.max().item():.4f}]")


PnL Loss Engine Initialized:
  * Price Codebook Tokens Cached : 512
  * PnL Lambda (Volatility Scale): 2.0
  * Directional Gamma (Sign Hinge): 1.0
  * Sample Body Return Min/Max   : [-0.0005, 0.0051]


---
## Stage 5: Baseline Zero-Shot Evaluation
Before executing fine-tuning, we measure the base model's performance on the multi-asset validation dataset to establish an empirical benchmark.


In [5]:
# Stage 5: Evaluate Baseline Performance Prior to Fine-Tuning
from klint.finetune.pnl_trainer import KlintPnLTrainer

baseline_trainer = KlintPnLTrainer(
    model=model,
    tokenizer=tokenizer,
    criterion=criterion,
    device=device,
)

print("Evaluating Baseline Foundation Model across multi-asset validation split...")
baseline_metrics = baseline_trainer.evaluate(val_loader, max_batches=30)

print("\n" + "=" * 55)
print("  BASELINE (PRE-TRAINED) VALIDATION RESULTS")
print("=" * 55)
print(f"  * Total Loss          : {baseline_metrics['val_loss_total']:.4f}")
print(f"  * Price Factor Loss   : {baseline_metrics['val_loss_price']:.4f}")
print(f"  * Directional Penalty : {baseline_metrics['val_loss_dir']:.4f}")
print(f"  * Directional Hit Rate: {baseline_metrics['val_hit_rate']*100:.2f}%")
print("=" * 55)


Evaluating Baseline Foundation Model across multi-asset validation split...

  BASELINE (PRE-TRAINED) VALIDATION RESULTS
  * Total Loss          : 8.8028
  * Price Factor Loss   : 9.1273
  * Directional Penalty : 0.0000
  * Directional Hit Rate: 41.00%


---
## Stage 6: Fine-Tuning Execution (Klint-32M -> Klint-32M v2)
We execute warm-start optimization using:
* **Optimizer**: AdamW ($\beta_1 = 0.9, \beta_2 = 0.95$, weight decay $= 0.01$)
* **Learning Rate**: $1 \times 10^{-4}$ decayed to $1 \times 10^{-6}$ via Cosine Annealing
* **Gradient Clipping**: $1.0$ norm
* **Target Output**: `./checkpoints/klint_32m_v2_release.pt`


In [6]:
# Stage 6: Run Fine-Tuning
TOTAL_STEPS = 500       # Adjust as needed (e.g. 1000 for full convergence)
EVAL_INTERVAL = 50      # Evaluate and check for best model every 50 steps
OUTPUT_BUNDLE = "checkpoints/klint_32m_v2_release.pt"

trainer = KlintPnLTrainer(
    model=model,
    tokenizer=tokenizer,
    criterion=criterion,
    learning_rate=1e-4,
    min_lr=1e-6,
    total_steps=TOTAL_STEPS,
    grad_clip=1.0,
    device=device,
)

print(f"Starting Fine-Tuning for {TOTAL_STEPS} steps...")
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    steps=TOTAL_STEPS,
    eval_interval=EVAL_INTERVAL,
    save_path=OUTPUT_BUNDLE,
)


Starting Fine-Tuning for 500 steps...
KLINT-32M v2 FINANCIAL FINE-TUNING ENGINE
Device: cuda | Total Steps: 500 | Eval Every: 50 steps
PnL Lambda: 2.0 | Directional Gamma: 1.0
Step   50/ 500 | Train Loss: 2.983 (Dir: 0.00) | Train HitRate: 49.2% | Val Loss: 2.843 | Val HitRate: 42.1% | LR: 9.76e-05 | 1.0 step/s
Step  100/ 500 | Train Loss: 2.850 (Dir: 0.00) | Train HitRate: 47.3% | Val Loss: 2.826 | Val HitRate: 43.4% | LR: 9.05e-05 | 0.9 step/s
Step  150/ 500 | Train Loss: 3.049 (Dir: 0.00) | Train HitRate: 42.1% | Val Loss: 2.823 | Val HitRate: 42.1% | LR: 7.96e-05 | 0.9 step/s
Step  200/ 500 | Train Loss: 2.784 (Dir: 0.00) | Train HitRate: 40.4% | Val Loss: 2.823 | Val HitRate: 43.0% | LR: 6.58e-05 | 0.9 step/s
Step  250/ 500 | Train Loss: 2.880 (Dir: 0.00) | Train HitRate: 45.7% | Val Loss: 2.819 | Val HitRate: 42.4% | LR: 5.05e-05 | 0.9 step/s
Step  300/ 500 | Train Loss: 2.977 (Dir: 0.00) | Train HitRate: 41.8% | Val Loss: 2.816 | Val HitRate: 42.6% | LR: 3.52e-05 | 0.9 step/s
St

---
## Stage 7: Performance Analytics & Candlestick Generation
We inspect the training trajectory and evaluate the upgraded **Klint-32M v2** model by generating future synthetic candlestick trajectories with invariant guarantees:
$$High \ge \max(Open, Close) \quad \text{and} \quad Low \le \min(Open, Close)$$


In [7]:
# Stage 7.1: Plot Training & Validation Trajectory
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Loss curves
axes[0].plot(history["step"], history["loss"], label="Train PnL Loss", color="#1f77b4", lw=2)
axes[0].plot(history["step"], history["val_loss"], label="Val PnL Loss", color="#ff7f0e", lw=2, marker="o")
axes[0].set_title("Loss Trajectory (PnL-Weighted)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Optimization Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Panel 2: Directional Hit Rate Progression
axes[1].plot(history["step"], [h * 100 for h in history["hit_rate"]], label="Train Hit Rate", color="#2ca02c", lw=2)
axes[1].plot(history["step"], [vh * 100 for vh in history["val_hit_rate"]], label="Val Hit Rate", color="#d62728", lw=2, marker="o")
axes[1].axhline(50.0, color="gray", linestyle="--", alpha=0.7, label="Random Guess (50%)")
axes[1].set_title("Directional Hit Rate (%)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Optimization Step")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Panel 3: Before vs. After Benchmark
final_val_loss = history["val_loss"][-1] if history["val_loss"] else 0.0
final_val_hr = (history["val_hit_rate"][-1] if history["val_hit_rate"] else 0.0) * 100
base_val_hr = baseline_metrics["val_hit_rate"] * 100

categories = ["Base (Pre-Trained)", "Klint-32M v2 (Fine-Tuned)"]
hr_values = [base_val_hr, final_val_hr]
colors = ["#7f7f7f", "#2ca02c"]

bars = axes[2].bar(categories, hr_values, color=colors, width=0.5)
axes[2].set_title("Validation Hit Rate Comparison", fontsize=13, fontweight="bold")
axes[2].set_ylabel("Directional Hit Rate (%)")
axes[2].set_ylim(40, max(hr_values) + 10)
axes[2].grid(True, alpha=0.3, axis="y")

for bar in bars:
    yval = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2.0, yval + 0.8, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


In [8]:
# Stage 7.2: Candlestick Forecast Verification with Upgraded Model
# Load the best fine-tuned v2 model
v2_bundle = torch.load(OUTPUT_BUNDLE, map_location=device, weights_only=False)
model.load_state_dict(v2_bundle["model_state_dict"])
model.eval()

print("Loaded fine-tuned Klint-32M v2 bundle.")
print("Generating 15 future market bars conditioned on recent market context...")

# Pick a prompt window from validation set
sample_batch = next(iter(val_loader))
prompt_tokens = sample_batch["inputs"][0:1, :180].to(device)  # First 60 bars

# Autoregressively sample 15 future bars (45 factor tokens)
with torch.no_grad():
    generated_tokens = model.generate_tokens(
        prompt_tokens,
        num_bars=15,
        temperature=0.8,
        top_k=40,
        top_p=0.90,
    )

    # De-interleave and decode into continuous factors
    p_gen, r_gen, a_gen = FactorTokenizer.deinterleave(generated_tokens)
    rec_price, rec_range, rec_activity = tokenizer.decode_tokens(p_gen, r_gen, a_gen)

# Reconstruct exact invariant OHLCV candles
from klint.data.factors import FactorStreams
factor_streams = FactorStreams(
    price_path=rec_price[0].detach().cpu().numpy(),
    range_shape=rec_range[0].detach().cpu().numpy(),
    activity=rec_activity[0].detach().cpu().numpy(),
    anchor_price=100.0,
)
forecast_ohlcv = decoder.decode(factor_streams)

print(f"\nGenerated {len(forecast_ohlcv)} OHLCV Candlesticks:")
print(f"First 5 Forecast Candles [Open, High, Low, Close, Vol]:")
for i in range(min(5, len(forecast_ohlcv))):
    row = forecast_ohlcv[i]
    print(f"  Bar {i+1:2d}: O={row[0]:.2f}, H={row[1]:.2f}, L={row[2]:.2f}, C={row[3]:.2f}, V={row[4]:.0f}")

# Verify 100% invariant enforcement
high_valid = np.all(forecast_ohlcv[:, 1] >= np.maximum(forecast_ohlcv[:, 0], forecast_ohlcv[:, 3]))
low_valid = np.all(forecast_ohlcv[:, 2] <= np.minimum(forecast_ohlcv[:, 0], forecast_ohlcv[:, 3]))
print(f"\nInvariant Validation Check:")
print(f"  * High >= max(Open, Close) : {high_valid} (100% Guaranteed)")
print(f"  * Low <= min(Open, Close)  : {low_valid} (100% Guaranteed)")


Loaded fine-tuned Klint-32M v2 bundle.
Generating 15 future market bars conditioned on recent market context...

Generated 75 OHLCV Candlesticks:
First 5 Forecast Candles [Open, High, Low, Close, Vol]:
  Bar  1: O=100.01, H=102.42, L=97.59, C=100.00, V=884
  Bar  2: O=99.98, H=102.44, L=97.10, C=100.02, V=150
  Bar  3: O=100.02, H=102.47, L=96.24, C=100.04, V=713
  Bar  4: O=100.03, H=102.48, L=97.14, C=100.07, V=522
  Bar  5: O=100.07, H=102.50, L=95.56, C=100.06, V=106

Invariant Validation Check:
  * High >= max(Open, Close) : True (100% Guaranteed)
  * Low <= min(Open, Close)  : True (100% Guaranteed)


---
## Stage 8: Hugging Face Model Hub Push (Optional)
To share your upgraded `klint_32m_v2_release.pt` checkpoint to your Hugging Face model repository ([akhverm/Klint-32M](https://huggingface.co/akhverm/Klint-32M)), run the cell below with your Hugging Face write token.


In [ ]:
# Stage 8: Upload Upgraded v2 Checkpoint to Hugging Face (Optional)
# from huggingface_hub import HfApi, login
#
# HF_TOKEN = "your_hf_write_token_here"  # Or use colab userdata: from google.colab import userdata; userdata.get('HF_TOKEN')
# login(token=HF_TOKEN)
#
# api = HfApi()
# api.upload_file(
#     path_or_fileobj="checkpoints/klint_32m_v2_release.pt",
#     path_in_repo="klint_32m_v2_release.pt",
#     repo_id="akhverm/Klint-32M",
#     repo_type="model",
# )
# print("Successfully uploaded Klint-32M v2 release checkpoint to Hugging Face!")
